In [ ]:
# input
pred_file = "./tmp/filtered_pred.tsv"
fasta_file = "./tmp/filtered.fasta"
seq_to_rep_file = "./tmp/filtered_entryId-repId.tsv"
# output
output_file = "data/human_potential_258.tsv"

In [4]:
import pandas as pd
from Bio import SeqIO

df = pd.read_table(seq_to_rep_file, header=None)
id2rep = dict(zip(df[0], df[1]))

df = pd.read_table(pred_file)

id2seq = dict()
id2info = dict()
for r in SeqIO.parse(fasta_file, "fasta"):
    id2seq[r.id] = str(r.seq)
    id2info[r.id] = {
        "name": " ".join(r.description.split(" UA=")[0].split()[1:]),
        "gene_name": r.description.split("GN=")[1] if "GN=" in r.description else "",
    }

In [5]:
records = []
for _, row in df.iterrows():
    seq_id = row['seq_id']
    af2_seq_id = f"AFDB:AF-{row['seq_id']}-F1"
    positions = [int(i) - 1 for i in row['pred_seq_num'].split(",")]
    info = id2info[af2_seq_id]
    seq = id2seq[af2_seq_id]

    records.append(
        {
            "seq_id": seq_id,
            "rep_id": id2rep[seq_id],
            "pred_num_resi": ",".join(
                [f"{i + 1}{seq[i]}" for i in positions]
            ),  # to seq num
            "proba": row['proba'],
            "plddt": row['plddt'],
            "metal_type": row['metal_type'],
            "metal_group_type": row['metal_group_type'],
            "len": len(seq),
            "name": info["name"],
            "gene_name": info["gene_name"],
        }
    )

df = pd.DataFrame(records).sort_values(by="len")
df.to_csv(output_file, sep="\t", index=None)